# Optimización de Pollard Rho para Logaritmo Discreto

## Configuración Inicial

In [2]:
import random
import itertools
import statistics
import math
import json
import os
from sympy import randprime, gcd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd

# Crear directorio para imágenes si no existe
if not os.path.exists('img'):
    os.makedirs('img')

# Configuración del espacio de búsqueda
BITS_PRIMOS = [10, 14, 18, 22, 24]
ITERACIONES_BUSQUEDA = 5

# Espacio de búsqueda
ESPACIO_BUSQUEDA = {
    "K_particiones": [8, 16, 20, 24, 32],
    "metodo_entropia": ["modulo", "shift_2", "shift_4", "shift_8"],
    "estrategia_iter": ["teske_aleatorio", "hibrido"]
}

## Funciones Auxiliares

In [3]:
def generar_parametros_dlp(bits):
    """Genera parámetros para el problema del logaritmo discreto"""
    p = randprime(2**(bits-1), 2**bits)
    N = p - 1
    g = random.randint(2, p - 2)
    x_real = random.randint(1, N - 1)
    h = pow(g, x_real, p)
    return p, N, g, h, x_real

def preparar_multiplicadores(K, N, g, h, p, estrategia):
    """Prepara los multiplicadores para cada partición"""
    ramas = []
    for i in range(K):
        if estrategia == "teske_aleatorio":
            c_i = random.randint(1, N - 1)
            d_i = random.randint(1, N - 1)
        else:  # Híbrido
            c_i = (i + 1) % N
            d_i = (i * 2) % N
        M_i = (pow(g, c_i, p) * pow(h, d_i, p)) % p
        ramas.append((M_i, c_i, d_i))
    return ramas

def step_optimizado(x, a, b, p, N, K, entropia, ramas):
    """Función de paso optimizada"""
    if entropia == "shift_8": i = (x >> 8) % K
    elif entropia == "shift_4": i = (x >> 4) % K
    elif entropia == "shift_2": i = (x >> 2) % K
    else: i = x % K

    M_i, c_i, d_i = ramas[i]
    return (x * M_i) % p, (a + c_i) % N, (b + d_i) % N

def pollard_rho_classico_steps(p, N, g, h):
    """Implementación del algoritmo clásico de Pollard Rho"""
    def step(x, a, b):
        if x % 3 == 0:
            return (x * h) % p, a, (b + 1) % N
        elif x % 3 == 1:
            return (x * x) % p, (2 * a) % N, (2 * b) % N
        else:
            return (x * g) % p, (a + 1) % N, b

    x, a, b = 1, 0, 0
    X, A, B = x, a, b
    pasos = 0
    max_pasos = int(math.sqrt(N)) * 15

    for _ in range(max_pasos):
        x, a, b = step(x, a, b)
        X, A, B = step(X, A, B)
        X, A, B = step(X, A, B)
        pasos += 1
        if x == X and (b - B) % N != 0:
            return pasos
    return max_pasos

## Ejecucion de la Busqueda

In [4]:
def ejecutar_busqueda_por_ratios():
    # Medir algoritmo clásico
    pasos_classico = {}
    print("Midendo algoritmo clásico...")
    for bits in BITS_PRIMOS:
        pasos_bits = []
        for _ in range(ITERACIONES_BUSQUEDA):
            p, N, g, h, _ = generar_parametros_dlp(bits)
            pasos = pollard_rho_classico_steps(p, N, g, h)
            pasos_bits.append(pasos)
        pasos_classico[str(bits)] = statistics.median(pasos_bits)
        print(f"Algoritmo clásico para {bits} bits: {pasos_classico[str(bits)]} pasos")

    # Configurar búsqueda
    claves, valores = zip(*ESPACIO_BUSQUEDA.items())
    combinaciones = [dict(zip(claves, v)) for v in itertools.product(*valores)]
    resultados = []

    print("\n Iniciando búsqueda de configuraciones...")
    print(f"Total de configuraciones a evaluar: {len(combinaciones)}")

    # Evaluar configuraciones
    for idx, config in enumerate(combinaciones):
        ratios_por_bit = []
        pasos_medios = {}
        valido = True

        for bits in BITS_PRIMOS:
            pasos_bits = []
            raiz_N = math.sqrt(2**bits)
            max_pasos = int(raiz_N) * 60

            for _ in range(ITERACIONES_BUSQUEDA):
                p, N, g, h, _ = generar_parametros_dlp(bits)
                ramas = preparar_multiplicadores(config["K_particiones"], N, g, h, p, config["estrategia_iter"])

                a0, b0 = random.randint(1, N-1), random.randint(1, N-1)
                x0 = (pow(g, a0, p) * pow(h, b0, p)) % p
                xT, aT, bT = x0, a0, b0
                xH, aH, bH = x0, a0, b0
                pasos = 0

                for _ in range(max_pasos):
                    xT, aT, bT = step_optimizado(xT, aT, bT, p, N, config["K_particiones"], config["metodo_entropia"], ramas)
                    pasos += 1
                    xH, aH, bH = step_optimizado(xH, aH, bH, p, N, config["K_particiones"], config["metodo_entropia"], ramas)
                    xH, aH, bH = step_optimizado(xH, aH, bH, p, N, config["K_particiones"], config["metodo_entropia"], ramas)
                    if xT == xH and (bT - bH) % N != 0:
                        pasos_bits.append(pasos)
                        break

            if not pasos_bits:
                valido = False
                break

            mediana_pasos = statistics.median(pasos_bits)
            pasos_medios[str(bits)] = mediana_pasos
            ratio = mediana_pasos / raiz_N
            ratios_por_bit.append(ratio)

        if not valido:
            continue

        score = sum(ratios_por_bit) / len(BITS_PRIMOS)
        resultados.append({
            "config": config,
            "score": score,
            "detalle_pasos": pasos_medios
        })

        print(f"[{idx+1}/{len(combinaciones)}] Score: {score:.4f} | Config: {config}")

    if not resultados:
        print("No se encontraron configuraciones válidas")
        return None, pasos_classico, resultados

    # Ordenar resultados
    resultados.sort(key=lambda x: x["score"])
    mejor = resultados[0]

    print("\n Mejor configuración encontrada:")
    print(json.dumps(mejor, indent=2))

    # Comparación con clásico
    print("\n Comparación con algoritmo clásico:")
    mejoras_porcentuales = {}
    for bits in BITS_PRIMOS:
        mejor_pasos = mejor["detalle_pasos"].get(str(bits), float('inf'))
        classico_pasos = pasos_classico.get(str(bits), float('inf'))
        if mejor_pasos != float('inf') and classico_pasos != float('inf'):
            mejora = (classico_pasos - mejor_pasos) / classico_pasos * 100
            mejoras_porcentuales[str(bits)] = mejora
            print(f"{bits} bits: {mejor_pasos} (vs {classico_pasos} clásicos) - Mejora: {mejora:.1f}%")

    return mejor, pasos_classico, resultados, mejoras_porcentuales

In [5]:
mejor_config, pasos_classico, resultados, mejoras_porcentuales = ejecutar_busqueda_por_ratios()

Midendo algoritmo clásico...
Algoritmo clásico para 10 bits: 390 pasos
Algoritmo clásico para 14 bits: 1830 pasos
Algoritmo clásico para 18 bits: 6390 pasos
Algoritmo clásico para 22 bits: 29265 pasos
Algoritmo clásico para 24 bits: 60030 pasos

 Iniciando búsqueda de configuraciones...
Total de configuraciones a evaluar: 40
[1/40] Score: 0.5404 | Config: {'K_particiones': 8, 'metodo_entropia': 'modulo', 'estrategia_iter': 'teske_aleatorio'}
[2/40] Score: 1.9179 | Config: {'K_particiones': 8, 'metodo_entropia': 'modulo', 'estrategia_iter': 'hibrido'}
[3/40] Score: 0.6006 | Config: {'K_particiones': 8, 'metodo_entropia': 'shift_2', 'estrategia_iter': 'teske_aleatorio'}
[4/40] Score: 1.2427 | Config: {'K_particiones': 8, 'metodo_entropia': 'shift_2', 'estrategia_iter': 'hibrido'}
[5/40] Score: 0.4019 | Config: {'K_particiones': 8, 'metodo_entropia': 'shift_4', 'estrategia_iter': 'teske_aleatorio'}
[6/40] Score: 1.8280 | Config: {'K_particiones': 8, 'metodo_entropia': 'shift_4', 'estrateg

## Generación de Gráficas

In [6]:
def generar_graficas(mejor, pasos_classico, resultados, mejoras_porcentuales):
    if not mejor:
        return

    bits = BITS_PRIMOS
    mejor_pasos = [mejor["detalle_pasos"].get(str(b), float('inf')) for b in bits]
    classico_pasos = [pasos_classico.get(str(b), float('inf')) for b in bits]
    mejoras = [mejoras_porcentuales.get(str(b), 0) for b in bits]

    # 1. Gráfica de comparación principal
    plt.figure(figsize=(12, 6))
    plt.plot(bits, classico_pasos, 'o-', color='blue', label='Algoritmo Clásico', linewidth=2, markersize=8)
    plt.plot(bits, mejor_pasos, 's-', color='green', label='Mejor Configuración', linewidth=2, markersize=8)
    plt.title('Comparación Directa: Pasos vs Tamaño de Primo', fontsize=14)
    plt.xlabel('Tamaño del Primo (bits)', fontsize=12)
    plt.ylabel('Número de Pasos', fontsize=12)
    plt.grid(True)
    plt.legend(fontsize=12)
    for i, bit in enumerate(bits):
        plt.text(bit, classico_pasos[i] + 50, f"{classico_pasos[i]}", ha='center', color='blue')
        plt.text(bit, mejor_pasos[i] - 50, f"{mejor_pasos[i]}", ha='center', color='green')
    plt.tight_layout()
    plt.savefig('img/comparacion_directa.png', dpi=300)
    plt.close()

    # 2. Gráfica de ratios
    plt.figure(figsize=(12, 6))
    ratios = [mejor_pasos[i] / classico_pasos[i] if classico_pasos[i] != float('inf') else float('inf')
              for i in range(len(bits))]
    plt.plot(bits, ratios, 'o-', color='purple', linewidth=2, markersize=8)
    plt.axhline(y=1, color='r', linestyle='--')
    plt.title('Ratio de Pasos: Mejor Configuración / Algoritmo Clásico', fontsize=14)
    plt.xlabel('Tamaño del Primo (bits)', fontsize=12)
    plt.ylabel('Ratio de Pasos', fontsize=12)
    plt.grid(True)
    for i, bit in enumerate(bits):
        if ratios[i] != float('inf'):
            plt.text(bit, ratios[i] + 0.05, f"{ratios[i]:.2f}", ha='center', color='purple')
    plt.tight_layout()
    plt.savefig('img/comparacion_ratios.png', dpi=300)
    plt.close()

    # 3. Gráfica de mejoras porcentuales
    plt.figure(figsize=(12, 6))
    plt.plot(bits, mejoras, 'o-', color='orange', linewidth=2, markersize=8)
    plt.title('Porcentaje de Mejora vs Tamaño de Primo', fontsize=14)
    plt.xlabel('Tamaño del Primo (bits)', fontsize=12)
    plt.ylabel('Porcentaje de Mejora (%)', fontsize=12)
    plt.grid(True)
    for i, bit in enumerate(bits):
        if mejoras[i] != 0:
            plt.text(bit, mejoras[i] + 2, f"{mejoras[i]:.1f}%", ha='center', color='orange')
    plt.tight_layout()
    plt.savefig('img/mejoras_porcentuales.png', dpi=300)
    plt.close()

    # 4. Distribución de scores
    plt.figure(figsize=(12, 6))
    scores = [res["score"] for res in resultados]
    plt.hist(scores, bins=20, color='cyan', edgecolor='black')
    plt.axvline(x=mejor["score"], color='red', linestyle='--', label=f'Mejor: {mejor["score"]:.4f}')
    plt.title('Distribución de Scores de Configuraciones', fontsize=14)
    plt.xlabel('Score (pasos/√N)', fontsize=12)
    plt.ylabel('Frecuencia', fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('img/distribucion_scores.png', dpi=300)
    plt.close()

    # 5. Comparación por método de entropía
    plt.figure(figsize=(12, 6))
    entropias = list(ESPACIO_BUSQUEDA["metodo_entropia"])
    scores_entropia = []
    for e in entropias:
        scores_e = [res["score"] for res in resultados if res["config"]["metodo_entropia"] == e]
        scores_entropia.append(scores_e)

    plt.boxplot(scores_entropia, labels=entropias)
    plt.title('Distribución de Scores por Método de Entropía', fontsize=14)
    plt.xlabel('Método de Entropía', fontsize=12)
    plt.ylabel('Score (pasos/√N)', fontsize=12)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('img/scores_por_entropia.png', dpi=300)
    plt.close()

    # 6. Comparación por estrategia iterativa
    plt.figure(figsize=(12, 6))
    estrategias = list(ESPACIO_BUSQUEDA["estrategia_iter"])
    scores_estrategia = []
    for s in estrategias:
        scores_s = [res["score"] for res in resultados if res["config"]["estrategia_iter"] == s]
        scores_estrategia.append(scores_s)

    plt.boxplot(scores_estrategia, labels=estrategias)
    plt.title('Distribución de Scores por Estrategia Iterativa', fontsize=14)
    plt.xlabel('Estrategia Iterativa', fontsize=12)
    plt.ylabel('Score (pasos/√N)', fontsize=12)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('img/scores_por_estrategia.png', dpi=300)
    plt.close()

    # 7. Heatmap de configuraciones
    config_data = []
    for res in resultados:
        config = res["config"]
        config_data.append({
            "K": config["K_particiones"],
            "Entropía": config["metodo_entropia"],
            "Estrategia": config["estrategia_iter"],
            "Score": res["score"]
        })

    df = pd.DataFrame(config_data)
    pivot_table = df.pivot_table(index="K", columns="Entropía", values="Score", aggfunc='median')

    plt.figure(figsize=(12, 6))
    sns.heatmap(pivot_table, annot=True, fmt=".3f", cmap="YlGnBu", linewidths=0.5)
    plt.title('Heatmap de Scores por Configuración', fontsize=14)
    plt.tight_layout()
    plt.savefig('img/heatmap_configuraciones.png', dpi=300)
    plt.close()

    # 8. Top configuraciones
    plt.figure(figsize=(12, 6))
    top_configs = resultados[:5]
    for res in top_configs:
        pasos = [res["detalle_pasos"].get(str(b), float('inf')) for b in bits]
        label = f'K={res["config"]["K_particiones"]}, {res["config"]["metodo_entropia"]}'
        plt.plot(bits, pasos, '--', alpha=0.6, label=label)

    plt.plot(bits, mejor_pasos, 's-', color='green', label='Mejor Configuración', linewidth=2, markersize=8)
    plt.plot(bits, classico_pasos, 'o-', color='blue', label='Algoritmo Clásico', linewidth=2, markersize=8)
    plt.title('Comparación de las 5 Mejores Configuraciones', fontsize=14)
    plt.xlabel('Tamaño del Primo (bits)', fontsize=12)
    plt.ylabel('Número de Pasos', fontsize=12)
    plt.legend(fontsize=10, bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('img/top_configuraciones.png', dpi=300)
    plt.close()

In [7]:
if mejor_config:
    generar_graficas(mejor_config, pasos_classico, resultados, mejoras_porcentuales)
    print("\n Todas las gráficas han sido generadas en la carpeta img/")

C:\Users\Rodri\AppData\Local\Temp\ipykernel_19776\2716024848.py:79: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(scores_entropia, labels=entropias)
C:\Users\Rodri\AppData\Local\Temp\ipykernel_19776\2716024848.py:96: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(scores_estrategia, labels=estrategias)



 Todas las gráficas han sido generadas en la carpeta img/


## Resultados y Conclusiones

In [8]:
if mejor_config:
    print(" Resumen de Resultados:")
    print(f"Mejor configuración: {mejor_config['config']}")
    print(f"Score promedio: {mejor_config['score']:.4f}")
    print("\n Comparación detallada:")
    for bits in BITS_PRIMOS:
        mejor_pasos = mejor_config["detalle_pasos"].get(str(bits), 'N/A')
        classico_pasos = pasos_classico.get(str(bits), 'N/A')
        mejora = mejoras_porcentuales.get(str(bits), 'N/A')
        print(f"{bits} bits: {mejor_pasos} pasos (vs {classico_pasos} clásicos) - Mejora: {mejora:.1f}%")
else:
    print("No se encontraron configuraciones válidas")

 Resumen de Resultados:
Mejor configuración: {'K_particiones': 8, 'metodo_entropia': 'shift_4', 'estrategia_iter': 'teske_aleatorio'}
Score promedio: 0.4019

 Comparación detallada:
10 bits: 14 pasos (vs 390 clásicos) - Mejora: 96.4%
14 bits: 51 pasos (vs 1830 clásicos) - Mejora: 97.2%
18 bits: 256 pasos (vs 6390 clásicos) - Mejora: 96.0%
22 bits: 560 pasos (vs 29265 clásicos) - Mejora: 98.1%
24 bits: 1638 pasos (vs 60030 clásicos) - Mejora: 97.3%
